## Setup

In [ ]:
# create a virtual environment and then run this cell
!python.exe -m pip install --upgrade pip
!pip install -r requirements.txt

In [142]:
# imports
import numpy as np
import random
import networkx
import time
from scipy.sparse import csr_array
from scipy.optimize import linear_sum_assignment

## PageRank

In [143]:
# function to generate random 2D graph of any given size with uniform edge cost 1

def generate_random_2D_graph(M: int) -> csr_array:
    G = np.zeros((M,M))
    for i in range(M):
        for j in range(M):
            if random.randint(0,1) == 1:
                G[i][j] = 1
    return csr_array(G)


In [144]:
# helper functions

# function to calculate ∑_k a[j][k]
def sum_for_M(A: csr_array, j: int) -> int:
    result = 0
    non_zeroes = A.nonzero()
    for i in range(len(non_zeroes[0])):
        if non_zeroes[0][i] == j:
            result += A[j, non_zeroes[1][i]]
            
    # for k in range(A.shape[0]):
    #     result += A[j][k]
    return result

# function to construct M from input matrix A
def construct_M(A: csr_array) -> csr_array:
    M = np.zeros((A.shape[0], A.shape[1]))
    
    for i in range(A.shape[0]):
        for j in range(A.shape[1]):
            local_sum = sum_for_M(A,j)
            # if local_sum is 0 then we don't want to divide by it
            if local_sum == 0:
                M[i][j] = 0
            else:
                M[i][j] = A[j,i]/sum_for_M(A,j)
    
    return csr_array(M)

# function to calculate l1-norm of r_new - r_old for convergence
def l1_norm_1D(v1: np.array, v2: np.array) -> int:
    vector = v2-v1
    result = 0
    for value in vector:
        result += abs(value)
    return result

def subtract_vectors(v1: np.array, v2: np.array) -> np.array:
    result = np.zeros(v1.shape[0])
    for i in range(v1.shape[0]):
        result[i] = v2[i] - v1[i]
    return result

# not necessary anymore 
# def edges_of_vertex_calculation(G: csr_array, vertex: int, beta: int, r_new: np.array):
#     result = 0
#     for i in range(G.shape[0]):
#         if G[i][vertex] == 0:
#             continue
#         result += (beta * (r_new[i]/outdeg(G,i)))
#     return result

# calculates the sum of all values of a vector
def vector_sum(vector: np.array) -> int:
    result = 0
    for v in vector:
        result += v
    return result

# calculates outdeg of a vertex; 
def outdeg(G: csr_array, v: int) -> int:
    result = 0
    for u in G.nonzero()[0]:
        if u == v:
            result += 1
    return result

# calculates indeg of a vertex; 
def indeg(G: csr_array, v: int) -> int:
    result = 0
    for row in G:
        if row[v] > 0:
            result += 1
    return result

# checks if two vectors are equal; currently not important
# def equal_vectors(v1: np.array, v2: np.array) -> bool:
#     if len(v1.shape) > 1 or len(v2.shape) > 1:
#         raise Exception("Input arrays must be 1 dimensional vectors.")
    
#     if v1.shape[0] != v2.shape[0]:
#         raise Exception("Both input vectors must be of equal size.")
    
#     for i in range(v1.shape[0]):
#         if v1[i] != v2[i]:
#             print(f"Found unequal values: {v1[i]} of first vector is not equal to {v2[i]} of second vector!")
#             return False
        
#     return True

# creates a prettier output string for the resulting vector r of the PageRank algorithm
def vector_toString(vector: np.array) -> str:
    s = "[\n"
    for i in range(vector.shape[0]):
        s += f" Vertex {i}: {vector[i]}"
        if i < vector.shape[0]-1:
            s += ","
        s += "\n"
    s += "]\n"
    return s

# constructs the Google Matrix from the sparse matrix M
def google_matrix(M: csr_array, beta: int) -> csr_array:
    if beta > 1 or beta < 0:
        raise Exception("Error: Invalid beta value. Must be between and including 0 and 1.")
    
    helper_matrix = np.empty((M.shape[0], M.shape[1]))
    helper_matrix[:] = 1/M.shape[0]
    
    return beta*M + (1-beta)*helper_matrix

# checks if the given vertex of the graph is a dead-end (outdegree of vertex = 0)
def is_dead_end(G: csr_array, vertex: int) -> bool:
    if outdeg(G, vertex) == 0:
        return True
    return False

# function to turn NetworkX PageRank to NumPy for accuracy measures with our custom PageRank implementation
def pagerank_to_array(pagerank: dict) -> np.array:
    items = [pagerank[i] for i in list(pagerank.keys())]
    return np.array(items)
    

def print_accuracy_score(r: np.array, networkx_r: np.array) -> float:
    if len(r) != len(networkx_r):
        raise Exception("Input vectors must be of equal size.")
    
    networkx_r = pagerank_to_array(networkx_r)
    sum_accuracy = 0
    for i in range(len(r)):
        if r[i] <= networkx_r[i]:
            sum_accuracy += (r[i]/networkx_r[i])
        else:
            sum_accuracy += (networkx_r[i]/r[i])
    
    print(f"Custom implementation is {((sum_accuracy/len(r))*100):.2f}% accurate compared to NetworkX.")

# def initialize_transportation_matrix(G: csr_array, probability: float) -> np.array:
#     t = np.empty(G.shape, dtype=float)
#     t[:] = probability
#     return t

In [145]:
# important functions for PageRank

def power_iteration(M: csr_array, epsilon: float = 1e-6) -> np.array:
    n = M.shape[0]
    r_old = np.empty((n), dtype=float)
    r_old[:] = 1/n
    r_new = r_old.copy()
    
    while(True):
        r_new = M @ r_old
        if subtract_then_l1_norm(r_old, r_new) < epsilon:
            #print("Reached convergence. Stopping Power Iteration.")
            break
        
        r_old = r_new.copy()
    
    return r_new

def personalized_pagerank(G: csr_array, teleportation: np.array, epsilon: float = 1e-6, beta: float = 0.85) -> np.array:
    if beta > 1:
        raise Exception("Error: Beta parameter must not be above 1.")
    elif beta < 0:
        raise Exception("Error: Beta paramter must not be less than 0.")
    
    n = G.shape[0]
    
    print("\n=== Current step: Constructing M ===\n")
    M = construct_M(G)
    r_new = np.zeros(n)
    print("\n=== Current step: Initializing r ===\n")
    for i in range(n):
        r_new[i] = 1/n
    #print(f"Graph G: \n{G}\n\nMatrix M: \n{M}\n\nBeta: {beta}\nEpsilon: {epsilon}\n")

    iter = 0
    
    while(True):
        print(f"\n=== Iteration {iter+1} ===\n")
        #print(f"=================================\n\nIteration: {iter}\n\n{r_new.transpose()}\n\n=================================")
        
        r_old = r_new.copy()
        
        r_new_hat = (beta*M) @ r_new
        
        D = vector_sum(r_new_hat)
        for v in range(n):
            r_new[v] = (r_new_hat[v] + (teleportation[v]*(1-D)))
        
        if l1_norm_1D(r_old, r_new) < epsilon:
            #print(f"PageRank algorithm converged. Stopping now.\n")
            break
        
        iter += 1

    return r_new

def google_pagerank(G: csr_array, epsilon: float = 1e-6, beta: float = 0.85) -> np.array:
    M = construct_M(G)
    G_M = google_matrix(M, beta=beta)
    
    return power_iteration(G_M, epsilon=epsilon)

def pagerank(G: csr_array, teleportation: np.array, epsilon: float = 1e-6, beta: float = 0.85) -> np.array:
    # check for dead-ends
    for i in range(G.shape[0]):
        if is_dead_end(G,i):
            #print(f"Dead-end found. Using general PageRank.")
            return personalized_pagerank(G, teleportation=teleportation, epsilon=epsilon, beta=beta)
    
    #print("No dead-ends found. Using Google PageRank.")
    return google_pagerank(G, epsilon=epsilon, beta=beta)

In [146]:
# test functions

def test_custom_pagerank(G: csr_array, teleportation: np.array, epsilon: float = 1e-6, beta: float = 0.85):
    start = time.time()
    r1 = pagerank(G, teleportation=teleportation, epsilon=epsilon, beta=beta)
    end = time.time()
    custom_time = end - start
    
    start = time.time()
    r2 = networkx.pagerank(networkx.Graph(G))
    end = time.time()
    networkx_time = end - start
    
    print(f"Got PageRank vector: \n{vector_toString(r1)}\n in {custom_time:.5f} seconds.\n")
    print(f"Got NetworkX PageRank vector: \n{vector_toString(pagerank_to_array(r2))}\n in {networkx_time:.5f} seconds.\n")
    print_accuracy_score(r1,r2)

In [147]:
# Testing PageRank on lecture example

# lecture example graph as sparse matrix
A = csr_array(np.array([[0,1,0,0,0], [1,0,1,0,0], [0,0,0,1,1], [0,0,0,0,0], [0,1,0,0,0]]))
t = np.array([1/A.shape[0] for x in range(A.shape[0])])

test_custom_pagerank(A, teleportation=t)


=== Current step: Constructing M ===


=== Current step: Initializing r ===


=== Iteration 1 ===


=== Iteration 2 ===


=== Iteration 3 ===


=== Iteration 4 ===


=== Iteration 5 ===


=== Iteration 6 ===


=== Iteration 7 ===


=== Iteration 8 ===


=== Iteration 9 ===


=== Iteration 10 ===


=== Iteration 11 ===


=== Iteration 12 ===


=== Iteration 13 ===


=== Iteration 14 ===


=== Iteration 15 ===

Got PageRank vector: 
[
 Vertex 0: 0.1958069857355091,
 Vertex 1: 0.3355714833684033,
 Vertex 2: 0.1958069857355091,
 Vertex 3: 0.13640727258028928,
 Vertex 4: 0.13640727258028928
]

 in 0.01027 seconds.

Got NetworkX PageRank vector: 
[
 Vertex 0: 0.11228740326958728,
 Vertex 1: 0.2904251934608255,
 Vertex 2: 0.2904251934608255,
 Vertex 3: 0.11228740326958728,
 Vertex 4: 0.19457480653917453
]

 in 0.00409 seconds.

Custom implementation is 72.75% accurate compared to NetworkX.


In [148]:
# Testing PageRank on a few random graphs with teleportation 1/n

for i in range(5):
    print(f"Test: {i+1}\n")
    G = generate_random_2D_graph(random.randint(0,50))
    t = np.array([1/G.shape[0] for x in range(G.shape[0])])
    test_custom_pagerank(G, teleportation=t)
    print("\n")

Test: 1

Got PageRank vector: 
[
 Vertex 0: 0.031355086400569485,
 Vertex 1: 0.04504309911522205,
 Vertex 2: 0.03479459861254094,
 Vertex 3: 0.030218294790846822,
 Vertex 4: 0.04055997467095548,
 Vertex 5: 0.03262787911014829,
 Vertex 6: 0.04331708785037947,
 Vertex 7: 0.028531855937396173,
 Vertex 8: 0.027331109835051545,
 Vertex 9: 0.04610062906026861,
 Vertex 10: 0.04390416557357049,
 Vertex 11: 0.045110880154298025,
 Vertex 12: 0.04365404566046573,
 Vertex 13: 0.034792392385760144,
 Vertex 14: 0.026828335319703894,
 Vertex 15: 0.026482433949456055,
 Vertex 16: 0.04413586388263976,
 Vertex 17: 0.038362865224739896,
 Vertex 18: 0.03383241277509686,
 Vertex 19: 0.03905702610312803,
 Vertex 20: 0.03329028600280435,
 Vertex 21: 0.04389277963521132,
 Vertex 22: 0.04670901026038688,
 Vertex 23: 0.03279879165501437,
 Vertex 24: 0.0386255223372809,
 Vertex 25: 0.036505056375980297,
 Vertex 26: 0.03213851732108378
]

 in 1.00090 seconds.

Got NetworkX PageRank vector: 
[
 Vertex 0: 0.0306182

In [149]:
# generate a random graph and test out some different beta values but keep teleportation at 1/n

G = generate_random_2D_graph(50)

beta = 0.5
while(beta < 0.9):
    print(f"\n============ Test with beta={beta:.2f} ============\n")
    test_custom_pagerank(G,teleportation=1/5,beta=beta)
    beta += 0.05


============ Test with beta=0.50 ============

Got PageRank vector: 
[
 Vertex 0: 0.019072331642459757,
 Vertex 1: 0.019223610186050892,
 Vertex 2: 0.018356820610233866,
 Vertex 3: 0.020622092866682918,
 Vertex 4: 0.019689387792306997,
 Vertex 5: 0.01841464721871189,
 Vertex 6: 0.018752316947045912,
 Vertex 7: 0.022115007201259997,
 Vertex 8: 0.021105519528057542,
 Vertex 9: 0.018403257749805405,
 Vertex 10: 0.01955990221479148,
 Vertex 11: 0.02140350712992591,
 Vertex 12: 0.02055071318083395,
 Vertex 13: 0.01853009995855011,
 Vertex 14: 0.0218749751040472,
 Vertex 15: 0.018354661665116425,
 Vertex 16: 0.021432789622717243,
 Vertex 17: 0.018672339460762452,
 Vertex 18: 0.02013093780048409,
 Vertex 19: 0.018531128581771677,
 Vertex 20: 0.022285150039833872,
 Vertex 21: 0.021430485801017023,
 Vertex 22: 0.018758612320518467,
 Vertex 23: 0.01933584474978788,
 Vertex 24: 0.02039169143123188,
 Vertex 25: 0.0206299603889908,
 Vertex 26: 0.023547719577164245,
 Vertex 27: 0.022024798174556593

In [150]:
# I will have to add some tests with different teleportations but tbh no idea what that should look like

## Network Alignment

In [151]:
# get graphs
G = networkx.read_gml("./pa3_data/G.gml", label=None)
H = networkx.read_gml("./pa3_data/H.gml", label=None)

# get adjacency matrices of the graphs
G_A = networkx.adjacency_matrix(G=G)
H_A = networkx.adjacency_matrix(G=H)

# get Kronecker Product of both matrices
K = np.kron(G_A, H_A)
# load similarities
s = np.loadtxt("./pa3_data/sim.txt", delimiter=',')

# normalize s
s = s/s.shape[0]
s = csr_array(s)

In [152]:
# IsoRank helper functions

# computes L1 norm of r_old and r_new for error tolerance checking
def l1_norm_2D(a1: np.array, a2: np.array) -> int:
    vector = a2-a1
    result = 0
    for row in vector:
        for col in row:
            result += abs(col)
    return result

# Personalized PageRank adjusted for Network Alignment to handle its adjacency matrix and similarity matrix
def networt_alignment_pagerank(K: csr_array, s: csr_array, beta: float = 0.85, epsilon: float = 1e-6):
    if beta > 1:
        raise Exception("Error: Beta parameter must not be above 1.")
    elif beta < 0:
        raise Exception("Error: Beta paramter must not be less than 0.")

    n = K.shape[0]*K.shape[1]
    
    print("\n=== Current step: Constructing M ===\n")
    M = construct_M(K)
    
    r_new = np.empty(K.shape, dtype=float)
    r_new[:] = 1/n

    iter = 0
    
    while(True):
        print(f"\n=== Iteration {iter+1} ===\n")
        #print(f"=================================\n\nIteration: {iter}\n\n{r_new.transpose()}\n\n=================================")
        
        r_old = r_new.copy()
        
        r_new_hat = (beta*M) @ r_new
        
        D = np.sum(r_new_hat)
        for g in range(r_new_hat.shape[0]):
            for h in range(r_new_hat.shape[1]):
                r_new[g][h] = (r_new_hat[g][h] + (s[g][h]*(1-D)))

        
        if l1_norm_2D(r_old, r_new) < epsilon:
            #print(f"PageRank algorithm converged. Stopping now.\n")
            break
        
        iter += 1

    return r_new



In [153]:
# IsoRank implementation
def isoRank(K: csr_array, s: csr_array, beta: float = 0.85, epsilon: float = 1e-6):
    t = networt_alignment_pagerank(K=K, s=s, beta=beta, epsilon=epsilon)
    matching = linear_sum_assignment(t)
    return matching

In [154]:
# get Assignment similarities t with PageRank on K with s as teleportation and beta = 0.85

# t = pagerank(K, teleportation=s, beta=0.85)
# t = networt_alignment_pagerank(K=K, s=s)


In [155]:
isoRank(K=K,s=s)


=== Current step: Constructing M ===


=== Iteration 1 ===


=== Iteration 2 ===


=== Iteration 3 ===


=== Iteration 4 ===


=== Iteration 5 ===


=== Iteration 6 ===


=== Iteration 7 ===


=== Iteration 8 ===


=== Iteration 9 ===


=== Iteration 10 ===


=== Iteration 11 ===


=== Iteration 12 ===


=== Iteration 13 ===


=== Iteration 14 ===


=== Iteration 15 ===


=== Iteration 16 ===


=== Iteration 17 ===


=== Iteration 18 ===


=== Iteration 19 ===


=== Iteration 20 ===


=== Iteration 21 ===


=== Iteration 22 ===


=== Iteration 23 ===


=== Iteration 24 ===


=== Iteration 25 ===


=== Iteration 26 ===


=== Iteration 27 ===


=== Iteration 28 ===


=== Iteration 29 ===


=== Iteration 30 ===


=== Iteration 31 ===


=== Iteration 32 ===


=== Iteration 33 ===


=== Iteration 34 ===


=== Iteration 35 ===


=== Iteration 36 ===


=== Iteration 37 ===


=== Iteration 38 ===


=== Iteration 39 ===


=== Iteration 40 ===


=== Iteration 41 ===


=== Iteration 42 ===


===

(array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
         13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
         26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
         39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
         52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
         65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
         78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
         91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103,
        104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116,
        117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129,
        130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142,
        143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155,
        156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168,
        169, 170, 171, 172, 173, 174, 175, 176, 177